# 04. Lecke - Eszközhasználati Tervezési Minta

Ebben a leckében megismerkedsz az AI-ügynökök számára a Microsoft Agent Framework (Python) segítségével használható **Eszközhasználati** tervezési mintával. Témáink:

- Függvényeszközök definiálása az `@tool` dekorátorral és típusos paraméterekkel
- Eszköz sémák biztosítása, hogy a modell tudja, mit csinál minden eszköz
- Az eszközvégrehajtás szabályozása az `approval_mode` segítségével
- **Strukturált kimenet** visszaadása Pydantic modelleken és a `response_format` használatával

A szcenárió egy **utazásszervező ügynök**, amely képes helyszíneket keresni, elérhetőséget ellenőrizni és repülőjáratok információit lekérni.


## Beállítás


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from pydantic import BaseModel
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
# Create the Azure AI Foundry provider
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Eszközök definiálása az @tool dekorátorral

Az `@tool` dekorátor egy egyszerű Python függvényt alakít át olyan eszközzé, amelyet egy ügynök hívhat meg.  
Kulcspontok:

- A **docstring** lesz az eszköz leírása, amelyet a modell lát.
- A **típusannotációk** (beleértve az `Annotated`-ot leírásokkal) határozzák meg az eszköz sémáját.
- Az `approval_mode` szabályozza, hogy a felhasználónak jóvá kell-e hagynia minden hívást, mielőtt az végrehajtódik.


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get available vacation destinations."""
    return ["Barcelona", "Paris", "Berlin", "Tokyo", "Sydney", "New York City"]


@tool(approval_mode="never_require")
def check_availability(
    destination: Annotated[str, "The destination to check"],
) -> str:
    """Check booking availability for a destination."""
    availability = {
        "Barcelona": "Available - 3 spots left",
        "Paris": "Available",
        "Berlin": "Sold out",
        "Tokyo": "Available - 1 spot left",
        "Sydney": "Available",
        "New York City": "Available",
    }
    return availability.get(destination, "Unknown destination")


@tool(approval_mode="never_require")
def get_flight_info(
    origin: Annotated[str, "Origin airport code"],
    destination: Annotated[str, "Destination airport code"],
) -> str:
    """Get flight information between two cities."""
    flights = {
        "LHR-BCN": "BA 2042, Departs 08:30, Arrives 11:45, $350",
        "LHR-CDG": "AF 1081, Departs 09:15, Arrives 11:30, $280",
        "LHR-NRT": "JL 044, Departs 11:00, Arrives 07:00+1, $890",
    }
    return flights.get(
        f"{origin}-{destination}",
        f"No direct flights from {origin} to {destination}",
    )

## Többeszközös ügynök létrehozása

Add át mind a három eszközt az ügyfélnek, hogy a modell bármelyiket meghívhassa, amelyre szüksége van a felhasználó kérdésének megválaszolásához.


In [ ]:
travel_tools = [get_destinations, check_availability, get_flight_info]

agent = await provider.create_agent(
    name="TravelToolAgent",
    instructions="You are a travel agent. Use the available tools to answer questions about destinations, availability, and flights.",
    tools=travel_tools,
)

response = await agent.run(
    "What destinations do you have? Which ones are still available?"
)
print(response)

## Strukturált kimenet eszközökkel

Ha a `response_format` értékét egy Pydantic modellre állítjuk, az ügynöknek egy jól típusosított JSON objektumot kell visszaadnia a szabad formátumú szöveg helyett. Ez hasznos, ha a következő kód programozottan szeretné feldolgozni az eredményt.


In [ ]:
class BookingRecommendation(BaseModel):
    destination: str
    available: bool
    flight_details: str
    estimated_cost: int


class TravelPlan(BaseModel):
    recommendations: list[BookingRecommendation]


structured_agent = await provider.create_agent(
    name="StructuredTravelAgent",
    instructions=(
        "You are a travel agent. Use the available tools to find destinations, "
        "check availability, and get flight info. Return structured results."
    ),
    tools=[get_destinations, check_availability, get_flight_info],
)

response = await structured_agent.run(
    "I want to fly from London Heathrow to somewhere warm in Europe. "
    "Check what's available."
)
if response:
    print(response)

## Eszköz jóváhagyási minták

A `@tool` `approval_mode` paramétere szabályozza, hogy az eszköz hívásai előtt szükséges-e emberi jóváhagyás:

| Mód | Viselkedés |
|---|---|
| `"never_require"` | Az eszköz automatikusan fut — nincs szükség felhasználói megerősítésre. |
| `"always_require"` | Minden hívást a felhasználónak jóvá kell hagynia, mielőtt végrehajtódik. |

Használja az `"always_require"` értéket olyan eszközöknél, amelyek mellékhatással járnak (pl. repülőjegy foglalása, hitelkártya terhelése), hogy az emberi beavatkozás megmaradjon.


In [ ]:
@tool(approval_mode="always_require")
def book_flight(
    origin: Annotated[str, "Origin airport code"],
    destination: Annotated[str, "Destination airport code"],
    passenger_name: Annotated[str, "Full name of the passenger"],
) -> str:
    """Book a flight for a passenger. Requires approval before executing."""
    return (
        f"Flight booked from {origin} to {destination} "
        f"for {passenger_name}. Confirmation #TRV-2024-{hash(passenger_name) % 10000:04d}"
    )


print("Tool name:", book_flight.name)
print("Approval mode:", book_flight.approval_mode)

## Összefoglaló

Ebben a leckében megtanultad, hogyan kell:

1. **Eszközöket definiálni** az `@tool` dekorátorral, típusos paraméterekkel és docstringekkel, amelyek az eszköz sémájaként szolgálnak.
2. **Több eszközt összeállítani**, hogy az ügynök sorban hívhassa meg őket bonyolultabb lekérdezések megválaszolásához.
3. **Strukturált kimenetet visszaadni** úgy, hogy egy Pydantic modellt adsz át `response_format`-ként.
4. **Irányítani az eszköz jóváhagyását** az `approval_mode` segítségével, hogy érzékeny műveleteknél emberi ellenőrzés legyen.

Ezek a minták alapot képeznek megbízható, éles környezetben használható ügynökök létrehozásához, amelyek biztonságosan kommunikálhatnak külső rendszerekkel.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Nyilatkozat**:
Ezt a dokumentumot a [Co-op Translator](https://github.com/Azure/co-op-translator) mesterséges intelligencia alapú fordító szolgáltatás segítségével fordítottuk. Bár igyekszünk a pontosságra, kérjük, vegye figyelembe, hogy az automatikus fordítások hibákat vagy pontatlanságokat tartalmazhatnak. Az eredeti dokumentum a saját nyelvén tekintendő hiteles forrásnak. Fontos információk esetén szakmai emberi fordítást javasolunk. Nem vállalunk felelősséget a fordítás használatából eredő félreértésekért vagy félreértelmezésekért.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
